Copyright (C) 2026 S.H.Tekur

This program is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or any later version.

This program is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.  See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with this program.  If not, see <https://www.gnu.org/licenses/>.

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import copy
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode
from PIL import Image
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# =====================================================================
# SETUP & CONFIGURATION
# =====================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# Update these paths to your local environment
ROOT_DIR = "fermi_surfaces"
MODEL_PATH = "fermi_surface_rs_classifier.pth"

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0


# =====================================================================
# TRANSFORMATIONS
# =====================================================================

class CircularMask:
    def __init__(self, radius_ratio=0.48, jitter=0.0, fill=0.5):
        self.radius_ratio = radius_ratio
        self.jitter = jitter
        self.fill = fill

    def __call__(self, img_tensor):
        # img_tensor: [1, H, W] (float32, 0–1)
        _, h, w = img_tensor.shape
        cy = h / 2 + random.uniform(-self.jitter, self.jitter) * h
        cx = w / 2 + random.uniform(-self.jitter, self.jitter) * w
        r = self.radius_ratio * min(h, w) * random.uniform(0.95, 1.05)

        yy, xx = torch.meshgrid(
            torch.arange(h, device=img_tensor.device),
            torch.arange(w, device=img_tensor.device),
            indexing="ij"
        )
        mask = (((yy - cy) ** 2 + (xx - cx) ** 2) <= r ** 2).float().unsqueeze(0)
        return img_tensor * mask + self.fill * (1.0 - mask)


class AdaptiveThreshold:
    """
    Grayscale → adaptive threshold → morphological smoothing → thinning.
    Output: [1, H, W] in {0.0 (arcs), fill_bg (background)}.
    """
    def __init__(self, block_size=31, C=5, fill_bg=0.5):
        self.block_size = block_size
        self.C = C
        self.fill_bg = fill_bg

    def __call__(self, img_tensor):
        # img_tensor: [1, H, W] float32 in [0,1] on any device
        img_np = img_tensor.squeeze(0).cpu().numpy()
        img_np = np.clip(img_np * 255.0, 0, 255).astype(np.uint8)  # [H, W], uint8

        # 1) Adaptive threshold
        try:
            binary_np = cv2.adaptiveThreshold(
                img_np, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                self.block_size,
                self.C
            )
        except Exception as e:
            print(f"adaptiveThreshold failed, using Otsu fallback: {e}")
            _, binary_np = cv2.threshold(
                img_np, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
            )

        # 2) Morphological smoothing
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary_np = cv2.morphologyEx(binary_np, cv2.MORPH_CLOSE, kernel)
        binary_np = cv2.medianBlur(binary_np, 3)

        # 3) Thinning (fallback version, only uses basic OpenCV ops)
        thinned_np = self._thin_lines_fallback(binary_np)

        # 4) Back to tensor [1, H, W] in {0, fill_bg}
        binary_tensor = torch.from_numpy(thinned_np).float().unsqueeze(0) / 255.0
        binary_tensor = torch.where(
            binary_tensor > 0.5,
            torch.tensor(self.fill_bg, dtype=torch.float32),
            torch.tensor(0.0, dtype=torch.float32),
        )
        return binary_tensor

    def _thin_lines_fallback(self, binary_np):
        """
        Simple iterative thinning using erode–dilate cycles.
        Produces thinner arcs and removes small blobs.
        """
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
        thinned = binary_np.copy()
        max_iter = 10

        for _ in range(max_iter):
            eroded = cv2.erode(thinned, kernel, iterations=1)
            dilated = cv2.dilate(eroded, kernel, iterations=1)
            if np.array_equal(thinned, dilated):
                break
            thinned = dilated

        return thinned
        

class RandomDilation:
    """
    Randomly dilates the black arcs to simulate thick experimental arcs.
    Works on binarized [1, H, W] tensors in {0.0, 0.5}.
    """
    def __init__(self, p=0.5, max_kernel=5):
        self.p = p
        # Pre-build valid odd kernel sizes up to max_kernel
        self.kernel_sizes = [k for k in [3, 5, 7, 9] if k <= max_kernel]
        if not self.kernel_sizes:
            self.kernel_sizes = [3]  # fallback

    def __call__(self, img_tensor):
        if random.random() > self.p:
            return img_tensor

        img_np = (img_tensor.squeeze(0).cpu().numpy() * 255).astype(np.uint8)

        # Arcs are black (0) → invert so arcs become white for dilation
        arcs_np = np.where(img_np < 64, 255, 0).astype(np.uint8)

        k = random.choice(self.kernel_sizes)  # Now correctly picks an int from the list
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        dilated_arcs = cv2.dilate(arcs_np, kernel, iterations=1)

        # Re-invert: dilated arcs → black (0), background → 0.5
        result_np = np.where(dilated_arcs > 128, 0, 127).astype(np.uint8)
        result_tensor = torch.from_numpy(result_np).float().unsqueeze(0) / 255.0

        return result_tensor

        
class RandomSpeckle:
    """Adds scattered black dots to simulate ARPES measurement noise."""
    def __init__(self, p=0.5, density=0.02):
        self.p = p
        self.density = density  # Fraction of pixels to blacken

    def __call__(self, img_tensor):
        if random.random() > self.p:
            return img_tensor

        noise_mask = torch.rand_like(img_tensor) < self.density
        img_tensor = img_tensor.clone()
        img_tensor[noise_mask] = 0.0  # Black speckles
        return img_tensor


class ChiralFermiDataset(Dataset):
    def __init__(self, root_dir, split='train', image_size=224):
        """
        root_dir: path to dataset root (containing train/val/test subfolders)
        split: 'train', 'val', or 'test'
        """
        self.data_dir = Path(root_dir) / split
        self.image_paths = []
        self.labels = []
        self.is_train = (split == 'train')
        self.image_size = image_size

        # Collect image paths and labels
        for class_dir in self.data_dir.iterdir():
            if class_dir.is_dir() and class_dir.name in ['R', 'S']:
                label = 0 if class_dir.name == 'R' else 1
                for img_path in class_dir.glob('*.*'):
                    if img_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
                        self.image_paths.append(img_path)
                        self.labels.append(label)

        print(f"{split} set: {len(self.image_paths)} images")

        # Deterministic preprocessing
        self.preprocess = transforms.Compose([
            transforms.Grayscale(num_output_channels=1),
            transforms.Resize((image_size, image_size), antialias=True),
            transforms.ToTensor(),                     # [1, H, W], 0–1
            CircularMask(radius_ratio=0.48, jitter=0.0, fill=0.5),
            AdaptiveThreshold(block_size=31, C=5),     # binarize + thin
        ])

    def __len__(self):
        return len(self.image_paths)

    
# =====================================================================
# TRAINING-TIME AUGMENTATIONS
# =====================================================================

    
    def train_augment(self, img):
        """
        img: Tensor [1, H, W] AFTER preprocessing (binarized).
        Returns (augmented_img, hflip_done, vflip_done).
        """
        # 1) Physics-preserving affine
        angle = random.uniform(0, 360)
        max_dx = int(0.06 * img.shape[2])
        max_dy = int(0.06 * img.shape[1])
        translate = [random.randint(-max_dx, max_dx),
                     random.randint(-max_dy, max_dy)]
        scale = random.uniform(0.92, 1.08)

        img = TF.affine(
            img, angle=angle, translate=translate, scale=scale,
            shear=[0.0, 0.0],
            interpolation=InterpolationMode.BILINEAR,
            fill=0.5
        )

        # 2) Label-aware flips
        hflip_done = False
        vflip_done = False
        if random.random() < 0.5:
            img = TF.hflip(img)
            hflip_done = True
        if random.random() < 0.5:
            img = TF.vflip(img)
            vflip_done = True

        # 3) Optional blur to soften lines slightly
        sigma_blur = random.uniform(0.8, 2.0)
        k_blur = int(2 * round(3 * sigma_blur) + 1)
        img = TF.gaussian_blur(img, kernel_size=[k_blur, k_blur],
                               sigma=[sigma_blur, sigma_blur])

        # 4) Re-apply circular mask to clean border
        img = CircularMask(radius_ratio=0.48, jitter=0.02, fill=0.5)(img)
        img = RandomDilation(p=0.5, max_kernel=3)(img)
        img = RandomSpeckle(p=0.5, density=0.02)(img)

        return img, hflip_done, vflip_done

    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        try:
            img_pil = Image.open(img_path).convert('L')
            img = self.preprocess(img_pil)  # [1, H, W], binarized
        except Exception as e:
            print(f"Preprocess failed for {img_path}: {e}")
            img = torch.zeros(1, self.image_size, self.image_size)

        if self.is_train:
            img, hflip_done, vflip_done = self.train_augment(img)
            if hflip_done:
                label = 1 - label
            if vflip_done:
                label = 1 - label

        # Repeat grayscale to 3 channels for ResNet18
        img = img.repeat(3, 1, 1)  # [3, H, W]
        # IMPORTANT: no normalization for binarized data

        return img, torch.tensor(label, dtype=torch.float32)


# =====================================================================
# CREATE DATA LOADERS
# =====================================================================
train_dataset = ChiralFermiDataset(root_dir=ROOT_DIR, split="train", image_size=IMAGE_SIZE)
val_dataset = ChiralFermiDataset(root_dir=ROOT_DIR, split="val", image_size=IMAGE_SIZE)
test_dataset = ChiralFermiDataset(root_dir=ROOT_DIR, split="test", image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Train:", len(train_dataset), "Val:", len(val_dataset), "Test:", len(test_dataset))


# =====================================================================
# MODEL DEFINITION & TRAINING LOGIC
# =====================================================================
def build_model(freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, 1)
    return model

def set_trainable_layers(model, freeze_backbone):
    for name, param in model.named_parameters():
        if 'fc' in name:
            param.requires_grad = True
        else:
            param.requires_grad = not freeze_backbone
    return model

def binary_accuracy_from_logits(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()
    return (preds == targets).float().mean().item()

class EarlyStopping:
    def __init__(self, patience=7, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float("inf")
        self.counter = 0

    def step(self, current_loss):
        if current_loss < self.best_loss - self.min_delta:
            self.best_loss = current_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_acc, total_samples = 0.0, 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_acc += binary_accuracy_from_logits(logits.detach(), labels) * batch_size
        total_samples += batch_size
    return total_loss / total_samples, total_acc / total_samples

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_acc, total_samples = 0.0, 0.0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        logits = model(images)
        loss = criterion(logits, labels)
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_acc += binary_accuracy_from_logits(logits, labels) * batch_size
        total_samples += batch_size
    return total_loss / total_samples, total_acc / total_samples, None, None

# =====================================================================
# TWO-STAGE TRAINING EXECUTION
# =====================================================================
NUM_EPOCHS_STAGE_1 = 8    
NUM_EPOCHS_STAGE_2 = 25   
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 7

model = build_model(freeze_backbone=True).to(device)
criterion = nn.BCEWithLogitsLoss()

best_val_loss = float("inf")
best_val_acc = 0.0
best_state = copy.deepcopy(model.state_dict())

def run_training_stage(model, stage_name, epochs, lr, freeze_backbone):
    global best_val_loss, best_val_acc, best_state
    
    model = set_trainable_layers(model, freeze_backbone)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    
    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    early_stopper = EarlyStopping(patience=PATIENCE, min_delta=1e-4)
    
    print(f"\n--- Starting {stage_name} ---")
    print(f"Trainable parameters: {sum(p.numel() for p in trainable_params)}")
    
    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, MODEL_PATH)

        if early_stopper.step(val_loss):
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

# Stage 1: Train Head Only
run_training_stage(model, "Stage 1: Head Only", NUM_EPOCHS_STAGE_1, LEARNING_RATE, freeze_backbone=True)

# Stage 2: Fine-Tune Whole Model
model.load_state_dict(best_state) # Load best head
run_training_stage(model, "Stage 2: Full Fine-Tuning", NUM_EPOCHS_STAGE_2, LEARNING_RATE / 5, freeze_backbone=False)

model.load_state_dict(best_state)
print(f"\nTraining complete. Best Val Loss: {best_val_loss:.4f}, Best Val Acc: {best_val_acc:.4f}")